# Configurar los orígenes de Power BI

Ejecuta la celda siguiente desde Jupyter **antes de abrir el PBIP en Power BI Desktop**. Los archivos se toman de `output/` del mismo proyecto.

Si Jupyter está abierto fuera de la carpeta principal o `notebooks/`, indica la ubicación del proyecto cuando se solicite. El código ajusta únicamente las rutas de los parámetros de origen existentes.


In [ ]:
from pathlib import Path
import os
import re

# Localizar la carpeta principal del proyecto.
BASE = Path.cwd().resolve()
if BASE.name.lower() == "notebooks":
    BASE = BASE.parent
if not (BASE / "powerbi" / "Caso_Analitica_Clientes.pbip").is_file():
    BASE = Path(input("Carpeta principal del proyecto: ").strip().strip('"')).expanduser().resolve()

PBIP = BASE / "powerbi" / "Caso_Analitica_Clientes.pbip"
MODELO = BASE / "powerbi" / "Caso_Analitica_Clientes.SemanticModel" / "definition" / "expressions.tmdl"
ARCHIVOS = {'pArchivoExcel': 'Modelo_Analitica_Clientes.xlsx', 'pArchivoRadar': 'RadarCliente.csv'}

if not PBIP.is_file() or not MODELO.is_file():
    raise FileNotFoundError("Selecciona la carpeta que contiene el proyecto y powerbi/.")
if os.name != "nt":
    raise OSError("La configuración de orígenes Power BI Desktop se realiza en Windows.")

for archivo in ARCHIVOS.values():
    if not (BASE / "output" / archivo).is_file():
        raise FileNotFoundError(f"Falta output/{archivo}.")

contenido = MODELO.read_text(encoding="utf-8")
nuevo = contenido
for parametro, archivo in ARCHIVOS.items():
    patron = (r'(?m)^(expression\s+' + re.escape(parametro)
              + r'\s*=\s*)"([^"]*)"(\s+meta\s+\[IsParameterQuery=true, Type="Text", IsParameterQueryRequired=true\])')
    coincidencia = re.search(patron, nuevo)
    if not coincidencia:
        raise ValueError(f"No se encontró el parámetro {parametro} en el modelo.")
    ruta = str((BASE / "output" / archivo).resolve())
    nuevo = nuevo[:coincidencia.start(2)] + ruta + nuevo[coincidencia.end(2):]

if nuevo != contenido:
    MODELO.write_text(nuevo, encoding="utf-8")
print("Orígenes de Power BI configurados correctamente.")
